In [ ]:
# Cell 1: install required dependencies (don't touch torch to avoid downgrade)
!pip install -q transformers peft accelerate gguf sentencepiece protobuf
print("✅ Dependencies installed")

In [ ]:
# Cell 2: confirm LoRA adapter path
import os

ADAPTER_PATH = "/kaggle/input/datasets/lin07077/lora-adapter/lora_adapter"

if os.path.exists(ADAPTER_PATH):
    print(f"✅ Adapter found: {ADAPTER_PATH}")
    print(os.listdir(ADAPTER_PATH))
else:
    print(f"❌ Path not found, searching for adapter_config.json ...")
    for root, dirs, files in os.walk("/kaggle/input"):
        if "adapter_config.json" in files:
            print(f"  → Found: {root}")
    print("Please update ADAPTER_PATH to the path found above")

In [ ]:
# Cell 3: merge LoRA adapter into base model (float16, CPU mode)
# Kaggle P100 CUDA kernel is incompatible with current PyTorch version (sm_60), so we use CPU merge
# Kaggle P100 has ~29GB CPU RAM, more than enough for a 7B float16 model (~14GB)
import torch, os, json
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
OUTPUT_DIR    = "/kaggle/working/merged_model"

print("📥 Loading base model (float16 + CPU)...")
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="cpu",
    low_cpu_mem_usage=True,   # load layer by layer to reduce peak memory
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)

print("🔗 Loading and merging LoRA adapter...")
model  = PeftModel.from_pretrained(base, ADAPTER_PATH)
merged = model.merge_and_unload()

print("💾 Saving merged model...")
os.makedirs(OUTPUT_DIR, exist_ok=True)
merged.save_pretrained(OUTPUT_DIR, safe_serialization=True, max_shard_size="5GB")
tokenizer.save_pretrained(OUTPUT_DIR)

# remove quantization_config if present — it causes errors during llama.cpp conversion
cfg_path = os.path.join(OUTPUT_DIR, "config.json")
with open(cfg_path) as f:
    cfg = json.load(f)
if cfg.pop("quantization_config", None):
    with open(cfg_path, "w") as f:
        json.dump(cfg, f, indent=2)
    print("🧹 Removed quantization_config")

print("✅ Merge complete, model saved to", OUTPUT_DIR)

In [ ]:
# Cell 4: clone llama.cpp (only install GGUF conversion deps, don't touch torch)
!git clone https://github.com/ggerganov/llama.cpp --depth=1
!pip install -q gguf sentencepiece protobuf
print("✅ llama.cpp cloned")

In [ ]:
# Cell 5: convert to GGUF f16 → quantize to Q4_K_M
#
# Disk strategy:
#   /kaggle/working  20GB limit, merged_model already takes ~14GB
#   /tmp             RAM disk (most of the 29GB RAM is free at this point), write 15GB f16 temp file
#   Flow: write f16 to /tmp → delete merged_model (free 14GB) → quantize Q4 to /kaggle/working → delete f16
import subprocess, os, shutil

OUTPUT_DIR = "/kaggle/working/merged_model"
GGUF_F16   = "/tmp/chef_lora_f16.gguf"          # write to RAM disk, doesn't use working space
GGUF_Q4    = "/kaggle/working/chef_lora_q4.gguf"

# Step A: HuggingFace → GGUF f16 (output to /tmp)
print("🔄 Converting to GGUF f16 (writing to /tmp)...")
result = subprocess.run(
    ["python", "llama.cpp/convert_hf_to_gguf.py", OUTPUT_DIR,
     "--outfile", GGUF_F16, "--outtype", "f16"],
    capture_output=True, text=True
)
print(result.stdout[-2000:])   # only print the tail to avoid flooding output
if result.returncode != 0:
    print("❌ Conversion failed:")
    print(result.stderr[-2000:])
    raise RuntimeError("GGUF conversion failed")
print(f"✅ GGUF f16: {os.path.getsize(GGUF_F16)/1e9:.1f} GB")

# Step B: delete merged_model to free up /kaggle/working space
print("🧹 Deleting merged_model to free disk space...")
shutil.rmtree(OUTPUT_DIR)
print(f"   /kaggle/working space freed")

# Step C: build llama-quantize
print("🔨 Building llama-quantize...")
subprocess.run(["cmake", "llama.cpp", "-B", "llama.cpp/build"], check=True)
subprocess.run(
    ["cmake", "--build", "llama.cpp/build", "--config", "Release",
     "-j4", "--target", "llama-quantize"],
    check=True
)

# Step D: Q4_K_M quantization (read from /tmp, write to /kaggle/working)
print("⚙️ Quantizing to Q4_K_M...")
subprocess.run(
    ["./llama.cpp/build/bin/llama-quantize", GGUF_F16, GGUF_Q4, "Q4_K_M"],
    check=True
)

# Step E: clean up /tmp temp file
os.remove(GGUF_F16)

size_gb = os.path.getsize(GGUF_Q4) / 1e9
print(f"✅ Quantization complete: chef_lora_q4.gguf ({size_gb:.1f} GB)")
print("📂 Go to Kaggle Output panel on the right → /kaggle/working/chef_lora_q4.gguf → click Download")

In [ ]:
# Cell 6: download instructions
# Kaggle doesn't support google.colab.files.download()
# Just find chef_lora_q4.gguf in the right-side Output panel and click Download
print("✅ All done!")
print("👉 Download: Kaggle Notebook → Output panel (right side) → /kaggle/working/chef_lora_q4.gguf → Download")